In [4]:
import os
import re
import csv
from collections import defaultdict

def parse_log_file_all_metrics(path):
    """
    从日志中解析所有 start to eval 段，
    并自动匹配最近一次迭代编号（数字行开头）作为 epoch。
    """
    records = []
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if line.strip().startswith("start to eval"):
            # === 1. 向上查找最近的 "N val1 val2 val3" 行 ===
            epoch = -1
            for j in range(i - 1, max(-1, i - 50), -1):  # 向上查50行以内
                m = re.match(r"^\s*(\d+)\s+[-+]?\d+\.\d+", lines[j])
                if m:
                    epoch = int(m.group(1))
                    break

            eval_metrics = {}

            # === 2. 提取 hit 系列 ===
            if i + 1 < len(lines) and lines[i + 1].startswith("hit20"):
                m = re.search(r"hit20\s+([0-9.+-eE]+)", lines[i + 1])
                if m:
                    eval_metrics["hit20"] = float(m.group(1))
            if i + 2 < len(lines) and lines[i + 2].startswith("hit50"):
                m = re.search(r"hit50\s+([0-9.+-eE]+)", lines[i + 2])
                if m:
                    eval_metrics["hit50"] = float(m.group(1))
            if i + 3 < len(lines) and lines[i + 3].startswith("hit100"):
                m = re.search(r"hit100\s+([0-9.+-eE]+)", lines[i + 3])
                if m:
                    eval_metrics["hit100"] = float(m.group(1))

            # === 3. 提取复合指标（支持科学计数法） ===
            if i + 4 < len(lines) and lines[i + 4].startswith("roc_auc"):
                nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", lines[i + 4])
                if len(nums) >= 5:
                    fields = ['roc_auc', 'pr_auc', 'f1', 'mrr_pess', 'mrr_opt']
                    result_dict = dict(zip(fields, map(float, nums[-5:])))
                    eval_metrics.update(result_dict)

            if eval_metrics:
                eval_metrics["epoch"] = epoch
                records.append(eval_metrics)

    return records


# ==== 参数 ====
dataset = "ogbl_citation2"
ratio = 0.02
cs = [1]
convergences = [1.0]
pos_ratios = [1.3, 1.4, 1.5]
seeds = [1, 2, 3, 4, 5]

log_base_dir = "."
output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)

ALL_METRICS = ["epoch", "hit20", "hit50", "hit100", "roc_auc", "pr_auc", "f1", "mrr_pess", "mrr_opt"]

# ==== 主逻辑 ====
rows = []
for c in cs:
    for conv in convergences:
        for pr in pos_ratios:
            for seed in seeds:
                fname = f"c{c}-conv{conv}-pos{pr}-s{seed}-r{ratio}.log"
                log_path = os.path.join(log_base_dir, fname)
                if not os.path.isfile(log_path):
                    print(f"[WARN] 缺失: {log_path}")
                    continue

                records = parse_log_file_all_metrics(log_path)
                if not records:
                    print(f"[WARN] 无有效指标: {log_path}")
                    continue

                for rec in records:
                    row = {
                        "c": c,
                        "convergence": conv,
                        "pos_ratio": pr,
                        "seed": seed,
                        "epoch": rec.get("epoch", -1)
                    }
                    for k in ALL_METRICS:
                        if k not in ["epoch"]:
                            row[k] = rec.get(k, 0.0) * 100  # 统一放大100倍
                    rows.append(row)

# ==== 写入 CSV ====
csv_path = os.path.join(output_dir, f"{dataset}_epochwise_result.csv")
header = ["c", "convergence", "pos_ratio", "seed"] + ALL_METRICS
with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    writer.writerows(rows)

print(f"[INFO] 共解析 {len(rows)} 条记录，写入完毕: {csv_path}")


[INFO] 共解析 225 条记录，写入完毕: ./results/ogbl_citation2_epochwise_result.csv


In [7]:
import pandas as pd
import numpy as np
from tabulate import tabulate

# === 读取数据 ===
csv_path = "./results/ogbl_citation2_epochwise_result.csv"
df = pd.read_csv(csv_path)

# 确保数据类型正确
df["mrr_pess"] = df["mrr_pess"].astype(float)
df["epoch"] = df["epoch"].astype(int)

# === 按 (c, convergence, pos_ratio, epoch) 分组求均值和标准差 ===
summary = (
    df.groupby(["c", "convergence", "pos_ratio", "epoch"])["mrr_pess"]
      .agg(["mean", "std"])
      .reset_index()
      .sort_values(["c", "convergence", "pos_ratio", "epoch"])
)

# === 展示结果表 ===
print("========= MRR_pess 随 Epoch 的统计结果 (mean ± std) =========")
table = summary[["c", "convergence", "pos_ratio", "epoch", "mean", "std"]]
print(tabulate(table, headers="keys", tablefmt="github", floatfmt=".4f"))


========= MRR_pess 随 Epoch 的统计结果 (mean ± std) =========
|    |      c |   convergence |   pos_ratio |   epoch |   mean |    std |
|----|--------|---------------|-------------|---------|--------|--------|
|  0 | 1.0000 |        1.0000 |      1.3000 |  4.0000 | 0.0473 | 0.0145 |
|  1 | 1.0000 |        1.0000 |      1.3000 |  9.0000 | 0.0626 | 0.0105 |
|  2 | 1.0000 |        1.0000 |      1.3000 | 14.0000 | 0.0748 | 0.0084 |
|  3 | 1.0000 |        1.0000 |      1.3000 | 19.0000 | 0.0730 | 0.0062 |
|  4 | 1.0000 |        1.0000 |      1.3000 | 24.0000 | 0.0972 | 0.0268 |
|  5 | 1.0000 |        1.0000 |      1.3000 | 29.0000 | 0.1125 | 0.0310 |
|  6 | 1.0000 |        1.0000 |      1.3000 | 34.0000 | 0.0997 | 0.0176 |
|  7 | 1.0000 |        1.0000 |      1.3000 | 39.0000 | 0.1032 | 0.0108 |
|  8 | 1.0000 |        1.0000 |      1.3000 | 44.0000 | 0.1023 | 0.0182 |
|  9 | 1.0000 |        1.0000 |      1.3000 | 49.0000 | 0.1130 | 0.0301 |
| 10 | 1.0000 |        1.0000 |      1.3000 | 54.0000 | 